Installing Libraries

In [ ]:
!pip install gensim
!pip install transformers
!pip install datasets
!pip install seqeval
!pip install tensorflow-addons
!pip install flair
!pip install nltk
!pip install torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 31.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=42485c05e82d1e3694d00419bb00126da1be763712f8717e5911fec3c047f723
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval
ERROR: Could not find a version that satisfies the requirement tensorflow-addons (from versions: none)
ERROR: No matching distribution found for tensorflow-addons
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 14.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.6 MB/s

In [ ]:
!pip install TorchCRF
!pip install tensorflow
!pip install tensorflow-hub

In [ ]:
import nltk

In [ ]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

Mount drive before importing data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Loading Data and path setting up

In [ ]:
Data_directory   = '/content/drive/MyDrive/CSC4182_assignment1/'
TRAIN_FILE = Data_directory + 'train.txt'
VAL_FILE   = Data_directory + 'val.txt'
TEST_FILE  = Data_directory + 'test.txt'


Created a function that read each file line by line and extract token and label into pairs and return two lists

In [ ]:
def load_conll(path):
    sents, labels = [], []
    toks, lbls = [], []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.rstrip()
            if line == '':
                if toks: sents.append(toks); labels.append(lbls)
                toks, lbls = [], []
            else:
                p = line.split()
                if len(p) == 2: toks.append(p[0]); lbls.append(p[1])
        if toks: sents.append(toks); labels.append(lbls)
    return sents, labels

train_sents, train_labels = load_conll(TRAIN_FILE)
val_sents,   val_labels   = load_conll(VAL_FILE)
test_sents,  test_labels  = load_conll(TEST_FILE)
print(f'Train={len(train_sents)}  Val={len(val_sents)}  Test={len(test_sents)}')

Train=4560  Val=4581  Test=4797


Label Enoding and vocabulary building



In [ ]:
import numpy as np
from collections import Counter

LABEL2ID  = {'O': 0, 'B-Disease': 1, 'I-Disease': 2, '<PAD>': 3}
ID2LABEL  = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = 4
PAD_LBL   = 3

counter = Counter(t.lower() for s in train_sents for t in s)
VOCAB    = ['<PAD>', '<UNK>'] + [w for w, c in counter.items() if c >= 2]
W2I      = {w: i for i, w in enumerate(VOCAB)}
VOCAB_SIZE = len(VOCAB)
EMBED_DIM  = 100
print(f'Vocab size: {VOCAB_SIZE}')

Vocab size: 5433


Creating tokenizer functions

In [ ]:
from nltk.tokenize import word_tokenize
from transformers import BertTokenizerFast

HF_TOK = BertTokenizerFast.from_pretrained('bert-base-uncased')

def ws_tokenize(words, labels):
    return words, [LABEL2ID.get(l, 0) for l in labels]

def nltk_tokenize(words, labels):
    new_toks = word_tokenize(' '.join(words))
    aligned  = []
    orig_idx, consumed = 0, 0
    for tok in new_toks:
        if orig_idx >= len(words):
            aligned.append(0); continue
        aligned.append(LABEL2ID.get(labels[orig_idx], 0) if consumed == 0 else 0)
        consumed += len(tok)
        if consumed >= len(words[orig_idx]): orig_idx += 1; consumed = 0
    return new_toks, aligned

def hf_tokenize(words, labels):
    enc      = HF_TOK(words, is_split_into_words=True, truncation=True, max_length=512)
    word_ids = enc.word_ids()
    toks     = HF_TOK.convert_ids_to_tokens(enc['input_ids'])
    aligned, prev = [], None
    for wid in word_ids:
        if wid is None: aligned.append(PAD_LBL)
        elif wid != prev: aligned.append(LABEL2ID.get(labels[wid], 0))
        else: aligned.append(PAD_LBL)
        prev = wid
    return toks, aligned

TOKENIZERS = {
    'Whitespace':    ws_tokenize,
    'NLTK':          nltk_tokenize,
    'BPE/WordPiece': hf_tokenize,
}

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

creating NERDataset

In [ ]:
import torch, os, gc, pickle
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

DISK_CACHE_DIR = '/content/dataset_cache'
os.makedirs(DISK_CACHE_DIR, exist_ok=True)


def save_to_disk(obj, path):
    with open(path, 'wb') as f: pickle.dump(obj, f, protocol=4)

def load_from_disk(path):
    with open(path, 'rb') as f: return pickle.load(f)


class NERDataset(Dataset):
    def __init__(self, sents, labels, tok_fn, tok_name='ws', split_name='train'):
        safe  = tok_name.lower().replace('/','_')
        path  = f'{DISK_CACHE_DIR}/static_{safe}_{split_name}.pkl'
        if os.path.exists(path):
            self.data = load_from_disk(path)
        else:
            self.data = []
            for s, l in zip(sents, labels):
                toks, lbl_ids = tok_fn(s, l)
                tok_ids = [W2I.get(t.lower(), W2I['<UNK>']) for t in toks]
                self.data.append((
                    torch.tensor(tok_ids,  dtype=torch.long),
                    torch.tensor(lbl_ids, dtype=torch.long)
                ))
            save_to_disk(self.data, path)

    def __len__(self):  return len(self.data)
    def __getitem__(self, i): return self.data[i]


def collate(batch):
    toks, lbls = zip(*batch)
    tok_pad = pad_sequence(toks, batch_first=True, padding_value=W2I['<PAD>'])
    lbl_pad = pad_sequence(lbls, batch_first=True, padding_value=PAD_LBL)
    mask    = tok_pad != W2I['<PAD>']
    return tok_pad, lbl_pad, mask


print('NERDataset defined (disk-cached).')

Device: cpu
NERDataset defined (disk-cached).


Downloaded glove

In [ ]:
import os, zipfile, urllib.request, numpy as np

GLOVE_RAW  = '/content/glove.6B.100d.txt'
GLOVE_NPY  = '/content/dataset_cache/glove_matrix.npy'

if not os.path.exists(GLOVE_RAW):
    print('Downloading GloVe...')
    urllib.request.urlretrieve('http://nlp.stanford.edu/data/glove.6B.zip', '/content/glove.zip')
    with zipfile.ZipFile('/content/glove.zip') as z:
        z.extract('glove.6B.100d.txt', '/content/')
    print('Downloaded.')
else:
    print('GloVe raw file already present.')


Downloaded.


Build embedding matrices

In [ ]:
import numpy as np, os, gc
from gensim.models import Word2Vec, FastText

os.makedirs(DISK_CACHE_DIR, exist_ok=True)

W2V_NPY   = f'{DISK_CACHE_DIR}/w2v_matrix.npy'
FT_NPY    = f'{DISK_CACHE_DIR}/ft_matrix.npy'
GLOVE_NPY = f'{DISK_CACHE_DIR}/glove_matrix.npy'


def build_matrix_from_gensim(gmodel):
    m = np.zeros((VOCAB_SIZE, EMBED_DIM), dtype=np.float32)
    for w, i in W2I.items():
        m[i] = gmodel.wv[w] if w in gmodel.wv else np.random.normal(0,.1,EMBED_DIM)
    return m


# Word2Vec
if os.path.exists(W2V_NPY):
    print('Loading Word2Vec matrix from disk...')
    W2V_MATRIX = np.load(W2V_NPY)
else:
    print('Training Word2Vec...')
    w2v = Word2Vec(train_sents, vector_size=EMBED_DIM, window=5, min_count=1, workers=4, epochs=10)
    W2V_MATRIX = build_matrix_from_gensim(w2v)
    np.save(W2V_NPY, W2V_MATRIX)
    del w2v; gc.collect()
    print(f'  Saved → {W2V_NPY}')
print(f'Word2Vec matrix: {W2V_MATRIX.shape}')


# FastText
if os.path.exists(FT_NPY):
    print('Loading FastText matrix from disk...')
    FT_MATRIX = np.load(FT_NPY)
else:
    print('Training FastText...')
    ft = FastText(train_sents, vector_size=EMBED_DIM, window=5, min_count=1,
                  workers=4, epochs=10, min_n=3, max_n=6)
    FT_MATRIX = build_matrix_from_gensim(ft)
    np.save(FT_NPY, FT_MATRIX)
    del ft; gc.collect()
    print(f'  Saved → {FT_NPY}')
print(f'FastText matrix: {FT_MATRIX.shape}')


#  GloVe
if os.path.exists(GLOVE_NPY):
    print('Loading GloVe matrix from disk...')
    GLOVE_MATRIX = np.load(GLOVE_NPY)
else:
    print('Parsing GloVe vectors...')
    glove_vecs = {}
    with open(GLOVE_RAW, encoding='utf-8') as f:
        for line in f:
            p = line.split()
            glove_vecs[p[0]] = np.array(p[1:], dtype=np.float32)
    GLOVE_MATRIX = np.zeros((VOCAB_SIZE, EMBED_DIM), dtype=np.float32)
    for w, i in W2I.items():
        GLOVE_MATRIX[i] = glove_vecs.get(w, np.random.normal(0,.1,EMBED_DIM))
    np.save(GLOVE_NPY, GLOVE_MATRIX)
    del glove_vecs; gc.collect()
    print(f'  Saved → {GLOVE_NPY}')
print(f'GloVe matrix: {GLOVE_MATRIX.shape}')


print('\nAll embedding matrices ready.')


Training Word2Vec...
  Saved → /content/dataset_cache/w2v_matrix.npy
Word2Vec matrix: (5433, 100)
Training FastText...
  Saved → /content/dataset_cache/ft_matrix.npy
FastText matrix: (5433, 100)
Parsing GloVe vectors...
  Saved → /content/dataset_cache/glove_matrix.npy
GloVe matrix: (5433, 100)

All embedding matrices ready.


Bi-LSTM CRF model creating

In [ ]:
import torch.nn as nn
from TorchCRF import CRF

CRF_LABELS = 3   # O, B-Disease, I-Disease

class BiLSTM_CRF(nn.Module):
    def __init__(self, embed_matrix, embed_dim, hidden=256, layers=2, drop=0.3, use_elmo=False):
        super().__init__()
        self.use_elmo = use_elmo
        if not use_elmo:
            t = torch.from_numpy(embed_matrix).float()
            self.emb = nn.Embedding.from_pretrained(t, freeze=False, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden, layers, bidirectional=True,
                            batch_first=True, dropout=drop if layers > 1 else 0)
        self.drop = nn.Dropout(drop)
        self.fc   = nn.Linear(hidden * 2, CRF_LABELS)
        self.crf  = CRF(CRF_LABELS)

    def _emit(self, x, mask):
        h = self.drop(x if self.use_elmo else self.emb(x))
        h, _ = self.lstm(h)
        return self.fc(self.drop(h))

    def forward(self, x, labels, mask):
        emit = self._emit(x, mask)
        lbl  = labels.clone(); lbl[~mask] = 0
        lbl  = lbl.clamp(0, CRF_LABELS - 1)

        return -self.crf(
            emit.transpose(0, 1),
            lbl.transpose(0, 1),
            mask=mask.transpose(0, 1).bool()
        ).mean()

    def decode(self, x, mask):
        emit = self._emit(x, mask)
        return self.crf.viterbi_decode(
            emit,
            mask=mask.bool()
        )


print('BiLSTM_CRF defined.')


BiLSTM_CRF defined.


Training and evaluating the utilities

In [ ]:
from torch.optim import AdamW
from seqeval.metrics import f1_score

def train_eval(model, train_dl, val_dl, test_dl, epochs=15, lr=1e-3, patience=3):
    opt = AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    best_f1, best_state, wait = 0.0, None, 0
    for ep in range(1, epochs + 1):
        model.train()
        for toks, lbls, mask in train_dl:
            toks, lbls, mask = toks.to(device), lbls.to(device), mask.to(device)
            opt.zero_grad()
            loss = model(toks, lbls, mask)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
        vf1 = get_f1(model, val_dl)
        if vf1 > best_f1:
            best_f1 = vf1; best_state = {k: v.clone() for k, v in model.state_dict().items()}; wait = 0
        else:
            wait += 1
            if wait >= patience: break
        print(f'  ep{ep:02d} val_f1={vf1:.4f}')
    model.load_state_dict(best_state)
    return get_f1(model, test_dl)

def get_f1(model, loader):
    model.eval()
    preds, golds = [], []
    with torch.no_grad():
        for toks, lbls, mask in loader:
            toks, mask = toks.to(device), mask.to(device)
            dec = model.decode(toks, mask)
            for pred, gold, m in zip(dec, lbls.numpy(), mask.cpu().numpy()):
                L = int(m.sum())
                preds.append([ID2LABEL.get(p,'O') for p in pred[:L]])
                golds.append([ID2LABEL.get(int(g),'O') for g in gold[:L]])
    return f1_score(golds, preds)

Loading elmo via TF-Hub and  defining embedding functions

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
import os, gc


ELMO_CACHE_DIR = '/content/elmo_cache'
os.makedirs(ELMO_CACHE_DIR, exist_ok=True)

print('Loading ELMo from TF-Hub (first run downloads ~360 MB)...')
elmo_module = hub.load('https://tfhub.dev/google/elmo/3')
elmo_fn     = elmo_module.signatures['tokens']
ELMO_DIM    = 1024
print('ELMo loaded. Output dim:', ELMO_DIM)


def elmo_embed_and_save(sentences, save_path, batch_size=16):

    all_embs = []
    for i in range(0, len(sentences), batch_size):
        batch     = sentences[i : i + batch_size]
        lengths   = [len(s) for s in batch]
        max_len   = max(lengths)
        padded    = [s + [''] * (max_len - len(s)) for s in batch]
        tokens_t  = tf.constant(padded,  dtype=tf.string)
        lengths_t = tf.constant(lengths, dtype=tf.int32)
        out       = elmo_fn(tokens=tokens_t, sequence_len=lengths_t)
        emb_np    = out['elmo'].numpy()
        for j, L in enumerate(lengths):
            all_embs.append(emb_np[j, :L, :].astype(np.float16))
        del out, emb_np, tokens_t, lengths_t
        gc.collect()


    arr = np.empty(len(all_embs), dtype=object)
    for i, e in enumerate(all_embs):
        arr[i] = e
    np.save(save_path, arr, allow_pickle=True)
    del arr, all_embs
    gc.collect()
    print(f'    saved → {save_path}')




Loading ELMo from TF-Hub (first run downloads ~360 MB)...
ELMo loaded. Output dim: 1024


ELMo embeddings — Whitespace tokenizer
(Computes embeddings, saves to disk, frees RAM before next cell.)

In [ ]:
import gc

tok_name = 'Whitespace'
tok_fn   = TOKENIZERS[tok_name]
safe_name = tok_name.lower().replace('/','_')

for split_name, sents, labels in [
    ('train', train_sents, train_labels),
    ('val',   val_sents,   val_labels),
    ('test',  test_sents,  test_labels),
]:
    print(f'  Computing {split_name} / {tok_name} ...')
    tokenized = [tok_fn(s, l)[0] for s, l in zip(sents, labels)]
    save_path = f'{ELMO_CACHE_DIR}/{safe_name}_{split_name}.npy'
    elmo_embed_and_save(tokenized, save_path)
    del tokenized
    gc.collect()

print('Whitespace done. RAM should be back to baseline ')


  Computing train / Whitespace ...
    saved → /content/elmo_cache/whitespace_train.npy
  Computing val / Whitespace ...
    saved → /content/elmo_cache/whitespace_val.npy
  Computing test / Whitespace ...
    saved → /content/elmo_cache/whitespace_test.npy
Whitespace done. RAM should be back to baseline 


ELMo embeddings — NLTK tokenizer
(Computes embeddings, saves to disk, frees RAM before next cell.)

In [ ]:
import gc

tok_name = 'NLTK'
tok_fn   = TOKENIZERS[tok_name]
safe_name = tok_name.lower().replace('/','_')

for split_name, sents, labels in [
    ('train', train_sents, train_labels),
    ('val',   val_sents,   val_labels),
    ('test',  test_sents,  test_labels),
]:
    print(f'  Computing {split_name} / {tok_name} ...')
    tokenized = [tok_fn(s, l)[0] for s, l in zip(sents, labels)]
    save_path = f'{ELMO_CACHE_DIR}/{safe_name}_{split_name}.npy'
    elmo_embed_and_save(tokenized, save_path)
    del tokenized
    gc.collect()

print('NLTK done. RAM should be back to baseline ')



  Computing train / NLTK ...
    saved → /content/elmo_cache/nltk_train.npy
  Computing val / NLTK ...
    saved → /content/elmo_cache/nltk_val.npy
  Computing test / NLTK ...
    saved → /content/elmo_cache/nltk_test.npy
NLTK done. RAM should be back to baseline 


ELMo embeddings — BPE/WordPiece tokenizer
(Computes embeddings, saves to disk, frees RAM before next cell.)

In [ ]:
import gc

tok_name = 'BPE/WordPiece'
tok_fn   = TOKENIZERS[tok_name]
safe_name = tok_name.lower().replace('/','_')

for split_name, sents, labels in [
    ('train', train_sents, train_labels),
    ('val',   val_sents,   val_labels),
    ('test',  test_sents,  test_labels),
]:
    print(f'  Computing {split_name} / {tok_name} ...')
    tokenized = [tok_fn(s, l)[0] for s, l in zip(sents, labels)]
    save_path = f'{ELMO_CACHE_DIR}/{safe_name}_{split_name}.npy'
    elmo_embed_and_save(tokenized, save_path)
    del tokenized
    gc.collect()

print('BPE/WordPiece done. All 3 tokenizers saved to disk. ')

  Computing train / BPE/WordPiece ...
    saved → /content/elmo_cache/bpe_wordpiece_train.npy
  Computing val / BPE/WordPiece ...
    saved → /content/elmo_cache/bpe_wordpiece_val.npy
  Computing test / BPE/WordPiece ...
    saved → /content/elmo_cache/bpe_wordpiece_test.npy
BPE/WordPiece done. All 3 tokenizers saved to disk. 


9d. ELMo Dataset (loads from disk) + BiLSTM_CRF_ELMo model

In [ ]:
import numpy as np

class ELMoDataset(Dataset):
   def __init__(self, split_name, sents, labels, tok_fn, tok_name):
        safe_name = tok_name.lower().replace('/','_')
        path      = f'{ELMO_CACHE_DIR}/{safe_name}_{split_name}.npy'
        emb_list  = np.load(path, allow_pickle=True)
        self.data = []
        for emb, (s, l) in zip(emb_list, zip(sents, labels)):
            _, lbl_ids = tok_fn(s, l)
            elen    = emb.shape[0]
            lbl_ids = (lbl_ids + [PAD_LBL] * elen)[:elen]
            self.data.append((
                torch.tensor(emb.astype(np.float32)),
                torch.tensor(lbl_ids, dtype=torch.long)
            ))
        del emb_list; gc.collect()

    def __len__(self):  return len(self.data)
    def __getitem__(self, i): return self.data[i]


def elmo_collate(batch):
    embs, lbls = zip(*batch)
    emb_pad = pad_sequence(embs, batch_first=True, padding_value=0.0)
    lbl_pad = pad_sequence(lbls, batch_first=True, padding_value=PAD_LBL)
    lengths = torch.tensor([e.shape[0] for e in embs])
    maxlen  = emb_pad.shape[1]
    mask    = torch.arange(maxlen).unsqueeze(0) < lengths.unsqueeze(1)
    return emb_pad, lbl_pad, mask


class BiLSTM_CRF_ELMo(nn.Module):
    def __init__(self, elmo_dim=1024, hidden=256, layers=2, drop=0.3):
        super().__init__()
        self.proj = nn.Linear(elmo_dim, hidden * 2)
        self.lstm = nn.LSTM(hidden * 2, hidden, layers, bidirectional=True,
                            batch_first=True, dropout=drop if layers > 1 else 0)
        self.drop = nn.Dropout(drop)
        self.fc   = nn.Linear(hidden * 2, CRF_LABELS)

        self.crf  = CRF(CRF_LABELS)

    def _emit(self, emb):
        h = self.drop(torch.relu(self.proj(emb)))
        h, _ = self.lstm(h)
        return self.fc(self.drop(h))           # (B, T, C)

    def forward(self, emb, labels, mask):
        emit = self._emit(emb)
        lbl  = labels.clone(); lbl[~mask] = 0
        lbl  = lbl.clamp(0, CRF_LABELS - 1)
        return -self.crf(
            emit.transpose(0, 1),
            lbl.transpose(0, 1),
            mask=mask.transpose(0, 1).bool()
        ).mean()

    def decode(self, emb, mask):
        emit = self._emit(emb)
        return self.crf.viterbi_decode(
            emit,
            mask=mask.bool()
        )


def train_eval_elmo(model, train_dl, val_dl, test_dl, epochs=15, lr=1e-3, patience=3):
    opt = AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    best_f1, best_state, wait = 0.0, None, 0
    for ep in range(1, epochs + 1):
        model.train()
        for emb, lbls, mask in train_dl:
            emb, lbls, mask = emb.to(device), lbls.to(device), mask.to(device)
            opt.zero_grad()
            loss = model(emb, lbls, mask)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
        vf1 = _elmo_f1(model, val_dl)
        if vf1 > best_f1:
            best_f1 = vf1; best_state = {k:v.clone() for k,v in model.state_dict().items()}; wait=0
        else:
            wait += 1
            if wait >= patience: break
        print(f'  ep{ep:02d} val_f1={vf1:.4f}')
    model.load_state_dict(best_state)
    return _elmo_f1(model, test_dl)


def _elmo_f1(model, loader):
    model.eval()
    preds, golds = [], []
    with torch.no_grad():
        for emb, lbls, mask in loader:
            emb, mask = emb.to(device), mask.to(device)
            dec = model.decode(emb, mask)
            for pred, gold, m in zip(dec, lbls.numpy(), mask.cpu().numpy()):
                L = int(m.sum())
                preds.append([ID2LABEL.get(p,'O') for p in pred[:L]])
                golds.append([ID2LABEL.get(int(g),'O') for g in gold[:L]])
    return f1_score(golds, preds)


print('ELMoDataset and BiLSTM_CRF_ELMo defined.')

BERT token classification

In [ ]:
from transformers import BertForTokenClassification
import pickle, os, gc

BERT_ID2L  = {0:'O', 1:'B-Disease', 2:'I-Disease'}
BERT_L2ID  = {v:k for k,v in BERT_ID2L.items()}
BERT_CACHE = '/content/bert_cache'
BERT_MAX_LEN = 128
os.makedirs(BERT_CACHE, exist_ok=True)


class BERTNERDataset(Dataset):

    def __init__(self, sents, labels, tok_fn, tok_name, split_name):
        safe = tok_name.lower().replace('/','_')
        path = f'{BERT_CACHE}/{safe}_{split_name}.pkl'
        if os.path.exists(path):
            print(f'    Loading BERT cache: {os.path.basename(path)}')
            self.items = load_from_disk(path)
        else:
            print(f'    Building BERT cache: {os.path.basename(path)}')
            self.items = []
            for s, l in zip(sents, labels):
                words, lbl_ids = tok_fn(s, l)
                str_lbls = [ID2LABEL.get(i,'O') for i in lbl_ids]
                # No padding here — truncate only, no padding='max_length'
                enc = HF_TOK(
                    words,
                    is_split_into_words=True,
                    truncation=True,
                    max_length=BERT_MAX_LEN,
                    return_tensors='pt'
                )
                word_ids = enc.word_ids()
                aligned, prev = [], None
                for wid in word_ids:
                    if wid is None: aligned.append(-100)
                    elif wid != prev:
                        lbl = str_lbls[wid] if wid < len(str_lbls) else 'O'
                        aligned.append(BERT_L2ID.get(lbl, 0))
                    else: aligned.append(-100)
                    prev = wid
                # Store as plain python lists — much lighter than padded tensors
                self.items.append({
                    'input_ids':      enc['input_ids'].squeeze(0).tolist(),
                    'attention_mask': enc['attention_mask'].squeeze(0).tolist(),
                    'token_type_ids': enc['token_type_ids'].squeeze(0).tolist(),
                    'labels':         aligned,
                })
            save_to_disk(self.items, path)
            print(f'    Saved.')

    def __len__(self):  return len(self.items)
    def __getitem__(self, i): return self.items[i]


# dynamic padding

def bert_collate(batch):

    max_len = max(len(b['input_ids']) for b in batch)
    input_ids, attention_mask, token_type_ids, labels = [], [], [], []
    for b in batch:
        pad_len = max_len - len(b['input_ids'])
        input_ids.append(      b['input_ids']      + [0]    * pad_len)
        attention_mask.append( b['attention_mask'] + [0]    * pad_len)
        token_type_ids.append( b['token_type_ids'] + [0]    * pad_len)
        labels.append(         b['labels']         + [-100] * pad_len)
    return {
        'input_ids':      torch.tensor(input_ids,      dtype=torch.long),
        'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
        'token_type_ids': torch.tensor(token_type_ids, dtype=torch.long),
        'labels':         torch.tensor(labels,         dtype=torch.long),
    }


def train_eval_bert(train_dl, val_dl, test_dl, epochs=5, lr=2e-5, patience=2):

    model = BertForTokenClassification.from_pretrained(
        'bert-base-uncased', num_labels=3,
        id2label=BERT_ID2L, label2id=BERT_L2ID,
        ignore_mismatched_sizes=True
    )

    model.gradient_checkpointing_enable()
    model = model.to(device)

    opt = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    best_f1, best_state, wait = 0.0, None, 0

    def bert_f1(dl):
        model.eval()
        preds, golds = [], []
        with torch.no_grad():
            for b in dl:
                ids   = b['input_ids'].to(device)
                amask = b['attention_mask'].to(device)
                ttype = b['token_type_ids'].to(device)
                logits = model(input_ids=ids, attention_mask=amask,
                               token_type_ids=ttype).logits
                pred   = torch.argmax(logits, -1).cpu().numpy()
                gold   = b['labels'].numpy()
                for pr, go in zip(pred, gold):
                    ps, gs = [], []
                    for p, g in zip(pr, go):
                        if g == -100: continue
                        ps.append(BERT_ID2L.get(int(p),'O'))
                        gs.append(BERT_ID2L.get(int(g),'O'))
                    if ps: preds.append(ps); golds.append(gs)
        return f1_score(golds, preds)

    for ep in range(1, epochs + 1):
        model.train()
        for b in train_dl:
            ids   = b['input_ids'].to(device)
            amask = b['attention_mask'].to(device)
            ttype = b['token_type_ids'].to(device)
            lbls  = b['labels'].to(device)
            opt.zero_grad()
            loss = model(input_ids=ids, attention_mask=amask,
                         token_type_ids=ttype, labels=lbls).loss
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        vf1 = bert_f1(val_dl)
        if vf1 > best_f1:
            best_f1 = vf1
            best_state = {k:v.clone() for k,v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience: break
        print(f'  ep{ep:02d} val_f1={vf1:.4f}')
    model.load_state_dict(best_state)
    f1 = bert_f1(test_dl)
    free_mem(model)
    return f1


# Pre build BERT datasets one at a time , save to disk, free RAM after each
print('Pre-building BERT datasets...')
for tok_name, tok_fn in TOKENIZERS.items():
    print(f'  Tokenizer: {tok_name}')
    for split_name, sents, labels in [
        ('train', train_sents, train_labels),
        ('val',   val_sents,   val_labels),
        ('test',  test_sents,  test_labels),
    ]:
        ds = BERTNERDataset(sents, labels, tok_fn, tok_name, split_name)
        del ds; gc.collect()
    print(f'    {tok_name} cached and freed.')
print('Done — all BERT datasets on disk.')

In [ ]:
import gc, torch

BS     = 32
EPOCHS = 10
results = {}   # all F1 scores collected here across cells 11a–11i

STATIC_EMBEDDINGS = {
    'Word2Vec': (W2V_MATRIX,   EMBED_DIM),
    'GloVe':    (GLOVE_MATRIX, EMBED_DIM),
    'FastText': (FT_MATRIX,    EMBED_DIM),
}

def free_mem(model=None):
    #Delete model, empty CUDA cache, force GC.
    if model is not None:
        del model
    gc.collect()
    torch.cuda.empty_cache()

print('Config ready.')


Word2Vec × all 3 tokenizers

In [ ]:

emb_name, (matrix, edim) = 'Word2Vec', STATIC_EMBEDDINGS['Word2Vec']
for tok_name, tok_fn in TOKENIZERS.items():
    print(f'\n▶ {emb_name} + {tok_name}')
    tr_ds = NERDataset(train_sents, train_labels, tok_fn, tok_name, 'train')
    va_ds = NERDataset(val_sents,   val_labels,   tok_fn, tok_name, 'val')
    te_ds = NERDataset(test_sents,  test_labels,  tok_fn, tok_name, 'test')
    tr = DataLoader(tr_ds, BS, shuffle=True,  collate_fn=collate)
    va = DataLoader(va_ds, BS, shuffle=False, collate_fn=collate)
    te = DataLoader(te_ds, BS, shuffle=False, collate_fn=collate)
    model = BiLSTM_CRF(matrix, edim).to(device)
    f1 = train_eval(model, tr, va, te, epochs=EPOCHS)
    results[(emb_name, tok_name)] = f1
    print(f'  ✔ TEST F1 = {f1:.4f}')
    del tr_ds, va_ds, te_ds, tr, va, te
    free_mem(model)
print('\nWord2Vec done:', {k[1]:round(v,4) for k,v in results.items() if k[0]=='Word2Vec'})



▶ Word2Vec + Whitespace
  ep01 val_f1=0.6069
  ep02 val_f1=0.6415
  ep03 val_f1=0.6180
  ep04 val_f1=0.6512
  ep05 val_f1=0.6624
  ep06 val_f1=0.6819
  ep07 val_f1=0.6817
  ep08 val_f1=0.6518
  ✔ TEST F1 = 0.6721

▶ Word2Vec + NLTK
  ep01 val_f1=0.5814
  ep02 val_f1=0.6490
  ep03 val_f1=0.6719
  ep04 val_f1=0.6432
  ep05 val_f1=0.6858
  ep06 val_f1=0.6839
  ep07 val_f1=0.6904
  ep08 val_f1=0.6622
  ep09 val_f1=0.6642
  ✔ TEST F1 = 0.6804

▶ Word2Vec + BPE/WordPiece


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: <PAD> seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


  ep01 val_f1=0.0251
  ep02 val_f1=0.0267
  ep03 val_f1=0.0281
  ep04 val_f1=0.0313
  ep05 val_f1=0.0322
  ep06 val_f1=0.0305
  ep07 val_f1=0.0320
  ✔ TEST F1 = 0.0360

Word2Vec done: {'Whitespace': np.float64(0.6721), 'NLTK': np.float64(0.6804), 'BPE/WordPiece': np.float64(0.036)}


GloVe × all 3 tokenizers

In [ ]:

emb_name, (matrix, edim) = 'GloVe', STATIC_EMBEDDINGS['GloVe']
for tok_name, tok_fn in TOKENIZERS.items():
    print(f'\n▶ {emb_name} + {tok_name}')
    tr_ds = NERDataset(train_sents, train_labels, tok_fn, tok_name, 'train')
    va_ds = NERDataset(val_sents,   val_labels,   tok_fn, tok_name, 'val')
    te_ds = NERDataset(test_sents,  test_labels,  tok_fn, tok_name, 'test')
    tr = DataLoader(tr_ds, BS, shuffle=True,  collate_fn=collate)
    va = DataLoader(va_ds, BS, shuffle=False, collate_fn=collate)
    te = DataLoader(te_ds, BS, shuffle=False, collate_fn=collate)
    model = BiLSTM_CRF(matrix, edim).to(device)
    f1 = train_eval(model, tr, va, te, epochs=EPOCHS)
    results[(emb_name, tok_name)] = f1
    print(f'  ✔ TEST F1 = {f1:.4f}')
    del tr_ds, va_ds, te_ds, tr, va, te
    free_mem(model)
print('\nGloVe done:', {k[1]:round(v,4) for k,v in results.items() if k[0]=='GloVe'})



▶ GloVe + Whitespace
  ep01 val_f1=0.4771
  ep02 val_f1=0.6115
  ep03 val_f1=0.6539
  ep04 val_f1=0.6725
  ep05 val_f1=0.6836
  ep06 val_f1=0.6994
  ep07 val_f1=0.7056
  ep08 val_f1=0.6979
  ep09 val_f1=0.7124
  ep10 val_f1=0.7065
  ✔ TEST F1 = 0.7091

▶ GloVe + NLTK
  ep01 val_f1=0.6374
  ep02 val_f1=0.6654
  ep03 val_f1=0.6990
  ep04 val_f1=0.7091
  ep05 val_f1=0.7099
  ep06 val_f1=0.7155
  ep07 val_f1=0.7115
  ep08 val_f1=0.7170
  ep09 val_f1=0.6990
  ep10 val_f1=0.7077
  ✔ TEST F1 = 0.7001

▶ GloVe + BPE/WordPiece
  ep01 val_f1=0.0175
  ep02 val_f1=0.0285
  ep03 val_f1=0.0279
  ep04 val_f1=0.0304
  ep05 val_f1=0.0313
  ep06 val_f1=0.0327
  ep07 val_f1=0.0314
  ep08 val_f1=0.0329
  ep09 val_f1=0.0340
  ep10 val_f1=0.0330
  ✔ TEST F1 = 0.0388

GloVe done: {'Whitespace': np.float64(0.7091), 'NLTK': np.float64(0.7001), 'BPE/WordPiece': np.float64(0.0388)}


FastText × all 3 tokenizers

In [ ]:
emb_name, (matrix, edim) = 'FastText', STATIC_EMBEDDINGS['FastText']
for tok_name, tok_fn in TOKENIZERS.items():
    print(f'\n▶ {emb_name} + {tok_name}')
    tr_ds = NERDataset(train_sents, train_labels, tok_fn, tok_name, 'train')
    va_ds = NERDataset(val_sents,   val_labels,   tok_fn, tok_name, 'val')
    te_ds = NERDataset(test_sents,  test_labels,  tok_fn, tok_name, 'test')
    tr = DataLoader(tr_ds, BS, shuffle=True,  collate_fn=collate)
    va = DataLoader(va_ds, BS, shuffle=False, collate_fn=collate)
    te = DataLoader(te_ds, BS, shuffle=False, collate_fn=collate)
    model = BiLSTM_CRF(matrix, edim).to(device)
    f1 = train_eval(model, tr, va, te, epochs=EPOCHS)
    results[(emb_name, tok_name)] = f1
    print(f'  ✔ TEST F1 = {f1:.4f}')
    del tr_ds, va_ds, te_ds, tr, va, te
    free_mem(model)
print('\nFastText done:', {k[1]:round(v,4) for k,v in results.items() if k[0]=='FastText'})



▶ FastText + Whitespace
  ep01 val_f1=0.1926
  ep02 val_f1=0.5837
  ep03 val_f1=0.6192
  ep04 val_f1=0.6396
  ep05 val_f1=0.6647
  ep06 val_f1=0.6460
  ep07 val_f1=0.6615
  ✔ TEST F1 = 0.6513

▶ FastText + NLTK
  ep01 val_f1=0.6172
  ep02 val_f1=0.6275
  ep03 val_f1=0.6031
  ep04 val_f1=0.6552
  ep05 val_f1=0.6709
  ep06 val_f1=0.6603
  ep07 val_f1=0.6469
  ep08 val_f1=0.6795
  ep09 val_f1=0.6692
  ep10 val_f1=0.6761
  ✔ TEST F1 = 0.6711

▶ FastText + BPE/WordPiece


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: <PAD> seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


  ep01 val_f1=0.0212
  ep02 val_f1=0.0245
  ep03 val_f1=0.0283
  ep04 val_f1=0.0271
  ep05 val_f1=0.0315
  ep06 val_f1=0.0290
  ep07 val_f1=0.0296
  ✔ TEST F1 = 0.0360

FastText done: {'Whitespace': np.float64(0.6513), 'NLTK': np.float64(0.6711), 'BPE/WordPiece': np.float64(0.036)}


ELMO x Whitespace

In [ ]:
tok_name, tok_fn = 'Whitespace', TOKENIZERS['Whitespace']
print(f'\n▶ ELMo + {tok_name}')
tr_ds = ELMoDataset('train', train_sents, train_labels, tok_fn, tok_name)
va_ds = ELMoDataset('val',   val_sents,   val_labels,   tok_fn, tok_name)
te_ds = ELMoDataset('test',  test_sents,  test_labels,  tok_fn, tok_name)
tr = DataLoader(tr_ds, 16, shuffle=True,  collate_fn=elmo_collate)
va = DataLoader(va_ds, 16, shuffle=False, collate_fn=elmo_collate)
te = DataLoader(te_ds, 16, shuffle=False, collate_fn=elmo_collate)
model = BiLSTM_CRF_ELMo(elmo_dim=ELMO_DIM).to(device)
f1 = train_eval_elmo(model, tr, va, te, epochs=EPOCHS)
results[('ELMo', tok_name)] = f1
print(f'  ✔ TEST F1 = {f1:.4f}')
del tr_ds, va_ds, te_ds, tr, va, te
free_mem(model)



▶ ELMo + Whitespace
  ep01 val_f1=0.6655
  ep02 val_f1=0.7094
  ep03 val_f1=0.7154
  ep04 val_f1=0.7313
  ep05 val_f1=0.7194
  ep06 val_f1=0.7218
  ✔ TEST F1 = 0.7393


ELMO x NLTK

In [ ]:
tok_name, tok_fn = 'NLTK', TOKENIZERS['NLTK']
print(f'\n▶ ELMo + {tok_name}')
tr_ds = ELMoDataset('train', train_sents, train_labels, tok_fn, tok_name)
va_ds = ELMoDataset('val',   val_sents,   val_labels,   tok_fn, tok_name)
te_ds = ELMoDataset('test',  test_sents,  test_labels,  tok_fn, tok_name)
tr = DataLoader(tr_ds, 16, shuffle=True,  collate_fn=elmo_collate)
va = DataLoader(va_ds, 16, shuffle=False, collate_fn=elmo_collate)
te = DataLoader(te_ds, 16, shuffle=False, collate_fn=elmo_collate)
model = BiLSTM_CRF_ELMo(elmo_dim=ELMO_DIM).to(device)
f1 = train_eval_elmo(model, tr, va, te, epochs=EPOCHS)
results[('ELMo', tok_name)] = f1
print(f'  ✔ TEST F1 = {f1:.4f}')
del tr_ds, va_ds, te_ds, tr, va, te
free_mem(model)


▶ ELMo + NLTK
  ep01 val_f1=0.6740
  ep02 val_f1=0.6608
  ep03 val_f1=0.7219
  ep04 val_f1=0.7151
  ep05 val_f1=0.7285
  ep06 val_f1=0.7412
  ep07 val_f1=0.7072
  ep08 val_f1=0.7305
  ep09 val_f1=0.7422
  ep10 val_f1=0.6917
  ✔ TEST F1 = 0.7424


ELMO x BPE/WordPiece

In [ ]:
tok_name, tok_fn = 'BPE/WordPiece', TOKENIZERS['BPE/WordPiece']
print(f'\n▶ ELMo + {tok_name}')
tr_ds = ELMoDataset('train', train_sents, train_labels, tok_fn, tok_name)
va_ds = ELMoDataset('val',   val_sents,   val_labels,   tok_fn, tok_name)
te_ds = ELMoDataset('test',  test_sents,  test_labels,  tok_fn, tok_name)
tr = DataLoader(tr_ds, 16, shuffle=True,  collate_fn=elmo_collate)
va = DataLoader(va_ds, 16, shuffle=False, collate_fn=elmo_collate)
te = DataLoader(te_ds, 16, shuffle=False, collate_fn=elmo_collate)
model = BiLSTM_CRF_ELMo(elmo_dim=ELMO_DIM).to(device)
f1 = train_eval_elmo(model, tr, va, te, epochs=EPOCHS)
results[('ELMo', tok_name)] = f1
print(f'  ✔ TEST F1 = {f1:.4f}')
del tr_ds, va_ds, te_ds, tr, va, te
free_mem(model)



▶ ELMo + BPE/WordPiece
  ep01 val_f1=0.0295
  ep02 val_f1=0.0239
  ep03 val_f1=0.0289
  ✔ TEST F1 = 0.0330


BERT x Whitespace

In [ ]:
tok_name, tok_fn = 'Whitespace', TOKENIZERS['Whitespace']
print(f'\n▶ BERT + {tok_name}')
tr_ds = BERTNERDataset(train_sents, train_labels, tok_fn, tok_name, 'train')
va_ds = BERTNERDataset(val_sents,   val_labels,   tok_fn, tok_name, 'val')
te_ds = BERTNERDataset(test_sents,  test_labels,  tok_fn, tok_name, 'test')
tr = DataLoader(tr_ds, 32, shuffle=True)
va = DataLoader(va_ds, 32, shuffle=False)
te = DataLoader(te_ds, 32, shuffle=False)
f1 = train_eval_bert(tr, va, te, epochs=5)
results[('BERT', tok_name)] = f1
print(f'  ✔ TEST F1 = {f1:.4f}')
del tr_ds, va_ds, te_ds, tr, va, te
free_mem()

BERT x NLTK

In [ ]:
tok_name, tok_fn = 'NLTK', TOKENIZERS['NLTK']
print(f'\n▶ BERT + {tok_name}')
tr_ds = BERTNERDataset(train_sents, train_labels, tok_fn, tok_name, 'train')
va_ds = BERTNERDataset(val_sents,   val_labels,   tok_fn, tok_name, 'val')
te_ds = BERTNERDataset(test_sents,  test_labels,  tok_fn, tok_name, 'test')
tr = DataLoader(tr_ds, 16, shuffle=True)
va = DataLoader(va_ds, 16, shuffle=False)
te = DataLoader(te_ds, 16, shuffle=False)
f1 = train_eval_bert(tr, va, te, epochs=5)
results[('BERT', tok_name)] = f1
print(f'  ✔ TEST F1 = {f1:.4f}')
del tr_ds, va_ds, te_ds, tr, va, te
free_mem()

BERT x BPE/WordPiece

In [ ]:
tok_name, tok_fn = 'BPE/WordPiece', TOKENIZERS['BPE/WordPiece']
print(f'\n▶ BERT + {tok_name}')
tr_ds = BERTNERDataset(train_sents, train_labels, tok_fn, tok_name, 'train')
va_ds = BERTNERDataset(val_sents,   val_labels,   tok_fn, tok_name, 'val')
te_ds = BERTNERDataset(test_sents,  test_labels,  tok_fn, tok_name, 'test')
tr = DataLoader(tr_ds, 16, shuffle=True)
va = DataLoader(va_ds, 16, shuffle=False)
te = DataLoader(te_ds, 16, shuffle=False)
f1 = train_eval_bert(tr, va, te, epochs=5)
results[('BERT', tok_name)] = f1
print(f'  ✔ TEST F1 = {f1:.4f}')
del tr_ds, va_ds, te_ds, tr, va, te
free_mem()
